# Hidden Embezzlement Investigation: Instructor Answer Key

This notebook is the instructor companion for the student lab:
- Hidden_Embezzlement_Investigation.ipynb

It provides:
1. Expected analytical checkpoints
2. Evidence pulled from phase-4 detector outputs
3. Suggested grading rubric

In [1]:
from pathlib import Path
import json
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

## 1) Load Core Data and Detector Reports

In [2]:
tx_path = Path('data/medici_transactions.csv')
benford_path = Path('data/serving/benford_analysis.json')
vendor_path = Path('data/serving/vendor_concentration_analysis.json')
dup_path = Path('data/serving/duplicate_transaction_analysis.json')
round_path = Path('data/serving/round_number_clustering_analysis.json')

for p in [tx_path, benford_path, vendor_path, dup_path, round_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p.resolve()}')

df = pd.read_csv(tx_path)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['debit_amount'] = pd.to_numeric(df['debit_amount'], errors='coerce').fillna(0.0)
df['branch'] = df['branch'].fillna('').astype(str).str.strip()
df['counterparty'] = df['counterparty'].fillna('').astype(str).str.strip()
df['debit_account'] = df['debit_account'].fillna('').astype(str).str.strip()
df['type'] = df['type'].fillna('').astype(str).str.strip()

benford = json.loads(benford_path.read_text())
vendor = json.loads(vendor_path.read_text())
dups = json.loads(dup_path.read_text())
round_cluster = json.loads(round_path.read_text())

print(f'Transactions: {len(df):,}')
print(f'Date range: {df['date'].min().date()} -> {df['date'].max().date()}')
print(f'Benford flagged groups: {benford['meta']['groups_flagged']}')
print(f'Vendor concentration anomalies: {vendor['meta']['flagged_count']}')
print(f'Duplicate alerts: {dups['meta']['flagged_count']}')
print(f'Round clustering alerts: {round_cluster['meta']['flagged_count']}')

Transactions: 85,000
Date range: 1390-01-01 -> 1440-12-31
Benford flagged groups: 100
Vendor concentration anomalies: 6
Duplicate alerts: 0
Round clustering alerts: 0


## 2) Expected Checkpoint: Florence Expense Focus

Students should narrow to Florence expense behavior and produce a ranked vendor table.

In [3]:
expense_types = {'operating_expense', 'recurring_operating_expense', 'vendor_payment'}
expense_accounts = {
    'Wages', 'Rent', 'Maintenance', 'Courier Services', 'Supplies',
    'Security', 'Travel', 'Entertainment', 'Marketing',
    'Depreciation', 'Miscellaneous Expense'
}

fl = df[df['branch'] == 'Florence'].copy()
fl_exp = fl[(fl['type'].isin(expense_types)) | (fl['debit_account'].isin(expense_accounts))].copy()
fl_exp = fl_exp[fl_exp['debit_amount'] > 0].copy()
fl_exp['vendor'] = fl_exp['counterparty'].replace('', '(blank counterparty)')

vendor_summary = (
    fl_exp.groupby('vendor', as_index=False)
    .agg(transactions=('debit_amount', 'count'), total_amount=('debit_amount', 'sum'))
)
vendor_summary['share'] = vendor_summary['total_amount'] / vendor_summary['total_amount'].sum()
vendor_summary = vendor_summary.sort_values('total_amount', ascending=False).reset_index(drop=True)

display(vendor_summary.head(15))
print('Florence expense rows:', len(fl_exp))
print('Florence expense total:', f"{fl_exp['debit_amount'].sum():,.2f}")

,vendor,transactions,total_amount,share
0,Mercato Maintenance Works,248,2199577.23,0.136009
1,Arno Lamp Oil Merchants,261,2120284.44,0.131106
2,Guildhall Security Company,257,1975651.98,0.122163
3,Santa Maria Scribes,241,1963349.16,0.121402
4,San Lorenzo Couriers,243,1956222.61,0.120961
5,Florentine Paperworks,247,1831857.29,0.113271
6,Ponte Vecchio Rent Office,249,1753266.14,0.108412
7,Signoria Utilities Office,208,1353255.88,0.083677
8,Florence Operations,252,1018852.87,0.063000


Florence expense rows: 2206
Florence expense total: 16,172,317.60


## 3) Expected Checkpoint: Benford Evidence

Students should cite first-digit anomaly evidence.

In [4]:
benford_df = pd.DataFrame(benford['flagged_groups'])
if not benford_df.empty:
    benford_view = benford_df[['dimension', 'group_key', 'severity', 'sample_count', 'mad', 'chi_squared']]
    benford_view = benford_view.sort_values(['severity', 'mad'], ascending=[True, False])
    display(benford_view.head(20))
else:
    print('No Benford anomalies present.')

,dimension,group_key,severity,sample_count,mad,chi_squared
0,Vendor/Counterparty,Transfer to Milan,HIGH,188,0.048800,44.0346
1,Debit Account,Accounts Payable,HIGH,6525,0.036332,946.3256
2,Vendor/Counterparty,Transfer to Florence,HIGH,199,0.036029,27.6815
3,Vendor/Counterparty,Inter-branch transfer to Rome,HIGH,1455,0.035070,169.2055
4,Debit Account,Due from Rome,HIGH,1672,0.034514,187.1761
5,Debit Account,Due from Naples,HIGH,1460,0.033842,149.7057
6,Vendor/Counterparty,Inter-branch transfer to Naples,HIGH,1460,0.033842,149.7057
7,Vendor/Counterparty,Inter-branch transfer to Venice,HIGH,1385,0.033731,149.2566
8,Vendor/Counterparty,Transfer to Rome,HIGH,217,0.033535,24.5315
9,Vendor/Counterparty,Transfer to London,HIGH,208,0.032862,19.8540


## 4) Expected Checkpoint: Vendor Concentration Evidence

Students should detect vendor over-concentration in one or more account categories.

In [5]:
vendor_alerts = pd.DataFrame(vendor['anomalies'])
if not vendor_alerts.empty:
    cols = ['severity', 'branch', 'period', 'debit_account', 'counterparty', 'metric_value', 'vendor_amount', 'group_total_amount']
    display(vendor_alerts[cols].sort_values('metric_value', ascending=False).head(20))
    print('High severity concentration findings:', (vendor_alerts['severity'] == 'HIGH').sum())
else:
    print('No vendor concentration anomalies present.')

,severity,branch,period,debit_account,counterparty,metric_value,vendor_amount,group_total_amount
0,HIGH,Geneva,1412-12,Courier Services,Ponte Vecchio Rent Office,0.539794,6784.74,12569.12
1,HIGH,London,1391-02,Security,Florentine Paperworks,0.369594,2257.49,6108.03
2,HIGH,London,1391-02,Security,Signoria Utilities Office,0.327181,1998.43,6108.03
3,HIGH,London,1391-02,Security,San Lorenzo Couriers,0.286759,1751.53,6108.03
4,HIGH,Geneva,1412-12,Courier Services,Florentine Paperworks,0.250541,3149.08,12569.12
5,MEDIUM,Geneva,1412-12,Courier Services,Guildhall Security Company,0.193658,2434.11,12569.12


High severity concentration findings: 5


## 5) Identify a Suspicious Supplier Candidate

Instructor note: A strong student answer should combine concentration share and contextual branch/account reasoning.

In [ ]:
if vendor_alerts.empty:
    suspicious_supplier = None
    print('No concentration anomalies to rank.')
else:
    ranked = vendor_alerts.sort_values(['severity', 'metric_value'], ascending=[True, False]).copy()
    suspicious_supplier = ranked.iloc[0]['counterparty']
    print('Top suspicious supplier candidate:', suspicious_supplier)
    display(ranked.head(10))

## 6) Estimate Fraud Amount and Date Range

Estimator used here:
- For each suspicious vendor/account period, expected normal share cap = 5%.
- Excess above that cap is treated as estimated suspicious amount.

In [ ]:
if suspicious_supplier is None:
    print('No supplier selected for fraud estimation.')
else:
    fl_exp['month'] = fl_exp['date'].dt.to_period('M').astype(str)
    target = fl_exp[fl_exp['vendor'] == suspicious_supplier].copy()

    monthly_account_total = (
        fl_exp.groupby(['month', 'debit_account'], as_index=False)
        .agg(group_total=('debit_amount', 'sum'))
    )

    suspect_monthly = (
        target.groupby(['month', 'debit_account'], as_index=False)
        .agg(vendor_amount=('debit_amount', 'sum'), tx_count=('id', 'count'))
    )

    suspect_monthly = suspect_monthly.merge(monthly_account_total, on=['month', 'debit_account'], how='left')
    suspect_monthly['expected_max'] = 0.05 * suspect_monthly['group_total']
    suspect_monthly['estimated_excess'] = (suspect_monthly['vendor_amount'] - suspect_monthly['expected_max']).clip(lower=0)

    estimated_fraud_total = suspect_monthly['estimated_excess'].sum()
    flagged_windows = suspect_monthly[suspect_monthly['estimated_excess'] > 0][['month', 'debit_account']]

    if flagged_windows.empty:
        print('No suspicious windows found for this supplier under the 5% baseline rule.')
    else:
        suspicious_tx = target.merge(flagged_windows, on=['month', 'debit_account'], how='inner')
        fraud_start = suspicious_tx['date'].min()
        fraud_end = suspicious_tx['date'].max()

        print('Suspicious supplier:', suspicious_supplier)
        print('Estimated suspicious amount:', f"{estimated_fraud_total:,.2f}")
        print('Estimated suspicious date range:', fraud_start.date(), 'to', fraud_end.date())
        print('Suspicious transactions in range:', len(suspicious_tx))

        display(suspect_monthly.sort_values('estimated_excess', ascending=False).head(20))
        display(suspicious_tx[['id', 'date', 'debit_account', 'vendor', 'debit_amount', 'description']].sort_values('date').head(30))

## 7) Suggested Grading Rubric (100 points)

| Category | Points | Full-credit expectation |
|---|---:|---|
| Data loading + hygiene | 10 | Correctly loads CSV and handles dates/amounts |
| Transaction exploration | 10 | Correct totals and branch/type summaries |
| Florence expense filtering | 15 | Clear logic for expense-only subset |
| Vendor aggregation | 10 | Correct vendor totals/shares |
| Benford analysis | 15 | Correct first-digit test and interpretation |
| Vendor concentration | 15 | Correct share thresholds and flagged vendors |
| Suspicious supplier selection | 10 | Evidence-based supplier choice |
| Fraud amount estimate | 10 | Transparent estimation logic and value |
| Fraud date range | 5 | Date range with supporting transactions |

## Instructor Notes
- Accept defensible alternate suspects if students justify with data.
- Reward clear assumptions and reproducibility over matching one exact number.
- Encourage critique of false-positive risk in Benford and concentration rules.